# BDFP footprint model — v2

Five targets from one image: **height, weight, age, sex, and left/right foot.**

Two changes from v1:

* **No data is discarded.** All 566 images train, duplicates included.
* **Folds are grouped by image content as well as by participant**, so every
  copy of a duplicated image lands in the same fold. Without that, the same
  JPEG sitting under two participant IDs would appear in train and test at
  once and the scores would be inflated.

Settings → Accelerator → **GPU T4 x2** (the P100 is too old for Kaggle's PyTorch).

## 0. Accelerator check

In [ ]:
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print(torch.cuda.get_device_name(0), f'sm_{cap[0]}{cap[1]}')
    assert cap >= (7, 0), 'GPU too old for this PyTorch build. Use GPU T4 x2.'
else:
    print('NO GPU — Settings -> Accelerator -> GPU T4 x2')


## 1. Locate the data

Two files are needed: the images, and **`footprint_images.csv`** — one row per
image, already carrying the `side` column and the `cv_group` used for folds.

In [ ]:
import os, glob, zipfile
ROOT = '/kaggle/input'
IMG_DIR = '/kaggle/working/images'
os.makedirs(IMG_DIR, exist_ok=True)
for z in glob.glob(ROOT + '/**/*.zip', recursive=True):
    print('extracting', os.path.basename(z)); zipfile.ZipFile(z).extractall(IMG_DIR)
if len(glob.glob(IMG_DIR + '/**/*.jpg', recursive=True)) == 0:
    c = [os.path.dirname(q) for q in glob.glob(ROOT + '/**/BDFP*.jpg', recursive=True)]
    IMG_DIR = max(set(c), key=c.count) if c else None

IMG_CSV = next(iter(glob.glob(ROOT + '/**/footprint_images.csv', recursive=True)), None)
LAB_CSV = next(iter(glob.glob(ROOT + '/**/footprint_labels_CORRECTED.csv', recursive=True)), None)

print('IMG_DIR :', IMG_DIR, '| images:', len(glob.glob(IMG_DIR + '/**/*.jpg', recursive=True)))
print('IMG_CSV :', IMG_CSV)
print('LAB_CSV :', LAB_CSV, '(fallback only)')
assert IMG_DIR, 'no images found'
assert IMG_CSV or LAB_CSV, 'need footprint_images.csv (preferred) or footprint_labels_CORRECTED.csv'


## 2. Load the per-image table

If `footprint_images.csv` is present it is used directly — `side` and
`cv_group` come straight from the file, no filename parsing and no hashing.

The fallback path rebuilds the same table from filenames plus the
per-participant CSV, for when only that file is available.

In [ ]:
import numpy as np, pandas as pd, re, hashlib

if IMG_CSV:
    df = pd.read_csv(IMG_CSV)
    files = {os.path.basename(q): q for q in
             glob.glob(IMG_DIR + '/**/*.*', recursive=True)}
    df['path'] = df.image.map(files)
    missing = int(df.path.isna().sum())
    if missing:
        print(f'WARNING: {missing} rows in the csv have no matching image file')
        print(df[df.path.isna()].image.head(10).tolist())
        df = df[df.path.notna()].reset_index(drop=True)
    extra = set(files) - set(df.image)
    if extra:
        print(f'NOTE: {len(extra)} image files are not listed in the csv and will be skipped')
    print('loaded per-image csv directly')
else:
    lab = pd.read_csv(LAB_CSV); lab['participant_id'] = lab.participant_id.astype(str).str.strip()
    class Union:
        def __init__(self): self.p = {}
        def find(self, x):
            self.p.setdefault(x, x)
            while self.p[x] != x: self.p[x] = self.p[self.p[x]]; x = self.p[x]
            return x
        def union(self, a, b):
            ra, rb = self.find(a), self.find(b)
            if ra != rb: self.p[ra] = rb
    def md5(q):
        h = hashlib.md5()
        with open(q, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
        return h.hexdigest()
    rows = []
    for q in sorted(glob.glob(IMG_DIR + '/**/*.jpg', recursive=True)):
        b = os.path.basename(q); m = re.match(r'(BDFP\d+)(?:_([LR]))?', b)
        if not m: continue
        r = lab[lab.participant_id == m.group(1)]
        if not len(r): continue
        r = r.iloc[0]
        rows.append(dict(image=b, path=q, participant_id=m.group(1), side=m.group(2) or 'U',
                         gender=r.gender, age=r.age, height_cm=r.height_cm,
                         weight_kg=r.weight_kg, subject_group=r.subject_group, _md5=md5(q)))
    df = pd.DataFrame(rows)
    u = Union()
    for _, r in df.iterrows():
        u.union(('i', r.image), ('p', r.participant_id))
        u.union(('p', r.participant_id), ('s', str(r.subject_group)))
        u.union(('i', r.image), ('m', r._md5))
    df['cv_group'] = pd.factorize(pd.Series([str(u.find(('i', i))) for i in df.image]))[0]
    dup = df.groupby('_md5').participant_id.nunique()
    df['is_duplicate'] = df._md5.isin(set(dup[dup > 1].index))
    print('rebuilt the per-image table from filenames')

if 'lab_known' not in df:
    df['lab_known'] = (~df[['gender','age','height_cm','weight_kg']].isna().any(axis=1)).astype(int)
for c_ in ['age','height_cm','weight_kg']:
    df[c_] = df[c_].fillna(df[c_].median())
df['gender'] = df.gender.fillna(df.gender.mode()[0])

df['y_gender']   = (df.gender == 'Male').astype(np.float32)
df['y_side']     = (df.side == 'R').astype(np.float32)
df['side_known'] = (df.side != 'U').astype(np.float32)

print(f'\nrows        : {len(df)}')
print('side        :', df.side.value_counts().to_dict())
print('sex         :', df.gender.value_counts().to_dict())
print('cv_groups   :', df.cv_group.nunique(), ' <- folds split on this')
print('duplicates  :', int(df.get("is_duplicate", pd.Series(False, index=df.index)).sum()), 'rows kept')
print('missing lbl :', int((df.lab_known == 0).sum()), 'rows kept (excluded from metrics only)')
bad = sum(1 for _, g in df.groupby('participant_id') if g.cv_group.nunique() > 1)
print('leak check  :', bad, '(participants split across folds - must be 0)')
df.head()


## 3. Settings

In [ ]:
import argparse
args = argparse.Namespace(out='/kaggle/working', cache='/kaggle/working/_prep',
                          arch='resnet50', epochs=40, lr=2e-4, bs=16, folds=5)
os.makedirs(args.out, exist_ok=True)


## 4. Pipeline code

Five heads now. The side head is trained only where the side is known — a masked loss — and a horizontal flip inverts the side label, so the augmentation stays valid.

In [ ]:
"""
BDFP footprint model - complete pipeline for Kaggle.

    python train_footprint_kaggle.py

Runs four phases end to end:
  1. MANIFEST   match images to CSV rows, hash-dedup, drop corrupted entries
  2. PREPROCESS crop to the sheet, blank the handwriting, resize keeping aspect
  3. TRAIN      ResNet50, four heads (sex, height, weight, age), grouped CV
  4. REPORT     every score beside its do-nothing baseline

Design notes worth knowing before you read the numbers:

* PREPROCESSING IS DELIBERATELY COARSE. Precise ink segmentation proved
  unreliable on these prints - measuring the same person's left and right foot
  disagreed by 15%. So this does NOT try to trace the footprint. It crops to
  the paper, removes the handwriting band, and preserves aspect ratio, then
  lets the network find the foot. Fewer assumptions, fewer ways to be wrong.

* ASPECT RATIO IS PRESERVED. A square resize squashes an A4 photo by 1.33x and
  destroys the length-to-breadth ratio, which is the main sex cue. That is the
  single biggest fix over the earlier MobileNetV2 run.

* DUPLICATE IMAGES ARE DROPPED. An unauthenticated Drive downloader gave the
  same file to several participants with different labels. Any image whose
  content appears under conflicting labels is removed - training on those
  teaches the model that one picture is both a 23-year-old man and a
  34-year-old woman.

* BASELINES ARE ALWAYS PRINTED. 48.57% looked like a low score rather than a
  broken one because nothing said the majority class was 57.1%.
"""

import os
import re
import glob
import json
import time
import hashlib
import argparse
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, roc_auc_score

DEV = "cuda" if torch.cuda.is_available() else "cpu"
REG = ["height_cm", "weight_kg", "age"]
IN_H, IN_W = 448, 320                 # close to A4's 1.414, keeps proportions
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)


# ------------------------------------------------------------------ phase 1
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def build_manifest(img_dir, csv_path):
    paths = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG"):
        paths += glob.glob(os.path.join(img_dir, "**", ext), recursive=True)
    paths = sorted(set(paths))
    if not paths:
        raise SystemExit(f"no images found under {img_dir}")

    lab = pd.read_csv(csv_path)
    lab["participant_id"] = lab.participant_id.astype(str).str.strip()

    rows, no_match = [], 0
    for p in paths:
        base = os.path.basename(p)
        m = re.match(r"(BDFP\d+)(?:_([LR]))?", base)
        if not m:
            no_match += 1
            continue
        pid, side = m.group(1), m.group(2) or "U"
        r = lab[lab.participant_id == pid]
        if not len(r):
            no_match += 1
            continue
        r = r.iloc[0]
        if pd.isna(r.gender) or any(pd.isna(r[c]) for c in REG):
            no_match += 1
            continue
        rows.append({"path": p, "image": base, "participant_id": pid, "side": side,
                     "subject_group": r.get("subject_group", pid),
                     "gender": r.gender,
                     **{c: float(r[c]) for c in REG}})

    df = pd.DataFrame(rows)
    print(f"[1/4] {len(paths)} files -> {len(df)} matched to a labelled CSV row "
          f"({no_match} unmatched)")

    # ---- prefer correctly-named _L/_R files when both namings are present ----
    if (df.side != "U").any() and (df.side == "U").any():
        dup = set(df[df.side != "U"].participant_id)
        before = len(df)
        df = df[(df.side != "U") | (~df.participant_id.isin(dup))]
        if before != len(df):
            print(f"      dropped {before - len(df)} sequence-named copies of "
                  f"files that also exist with correct _L/_R names")

    # ---- content hashing: find one image serving several participants ----
    print("      hashing for duplicates ...")
    df["md5"] = df.path.apply(md5)

    # Step A. Where the same bytes appear under both a correct _L/_R name and a
    # sequence name, the _L/_R one came from the CSV's own Drive link and is
    # authoritative. Drop the sequence-named copy rather than both.
    drop = set()
    for h, sub in df.groupby("md5"):
        if len(sub) > 1 and (sub.side != "U").any() and (sub.side == "U").any():
            drop |= set(sub[sub.side == "U"].index)
    if drop:
        df = df.drop(index=list(drop))
        print(f"      resolved {len(drop)} name collisions in favour of _L/_R names")

    # Step B. Whatever conflicts remain are genuine download corruption: one
    # file served for several different people. Drop only the offending IMAGES,
    # not the participants - their other feet may still be fine.
    g = df.groupby("md5").participant_id.nunique()
    conflict = set()
    for h in g[g > 1].index:
        sub = df[df.md5 == h]
        if len(sub[["gender"] + REG].drop_duplicates()) > 1:
            conflict.add(h)
    before = len(df)
    df = df[~df.md5.isin(conflict)]
    print(f"      {len(conflict)} images had conflicting labels -> "
          f"dropped {before - len(df)} images")

    # exact duplicates within one participant (same file saved twice)
    before = len(df)
    df = df.drop_duplicates(subset=["participant_id", "md5"])
    if before != len(df):
        print(f"      removed {before - len(df)} exact duplicate files")

    df["y_gender"] = (df.gender == "Male").astype(np.float32)
    print(f"      FINAL: {len(df)} images, {df.subject_group.nunique()} subjects, "
          f"{df.gender.value_counts().to_dict()}")
    return df.reset_index(drop=True)


# ------------------------------------------------------------------ phase 2
def paper_crop(bgr):
    """Bounding box of the sheet. Coarse on purpose - no ink segmentation."""
    h, w = bgr.shape[:2]
    s = 700 / max(h, w)
    sm = cv2.resize(bgr, None, fx=s, fy=s)
    lab = cv2.cvtColor(sm, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    chroma = np.sqrt((A.astype(np.float32) - 128) ** 2 + (B.astype(np.float32) - 128) ** 2)
    paper = ((L > 110) & (chroma < 32)).astype(np.uint8) * 255
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    paper = cv2.morphologyEx(paper, cv2.MORPH_CLOSE, k, 3)
    paper = cv2.morphologyEx(paper, cv2.MORPH_OPEN, k, 2)
    n, lb, st, _ = cv2.connectedComponentsWithStats(paper, 8)
    if n <= 1:
        return bgr
    i = 1 + int(np.argmax(st[1:, 4]))
    if st[i, 4] < 0.25 * paper.size:
        return bgr
    x, y, bw, bh, _ = st[i]
    x, y, bw, bh = [int(v / s) for v in (x, y, bw, bh)]
    return bgr[max(y, 0):min(y + bh, h), max(x, 0):min(x + bw, w)]


def preprocess(bgr):
    sheet = paper_crop(bgr)
    if sheet.size == 0:
        sheet = bgr
    h, w = sheet.shape[:2]
    sheet = sheet[int(0.15 * h):, :]              # blank the handwriting band
    h, w = sheet.shape[:2]
    s = min(IN_H / h, IN_W / w)                   # aspect preserved
    nh, nw = max(int(h * s), 1), max(int(w * s), 1)
    canvas = np.full((IN_H, IN_W, 3), 255, np.uint8)
    t, l = (IN_H - nh) // 2, (IN_W - nw) // 2
    canvas[t:t + nh, l:l + nw] = cv2.resize(sheet, (nw, nh), interpolation=cv2.INTER_AREA)
    return canvas


def cache_all(df, cache_dir):
    os.makedirs(cache_dir, exist_ok=True)
    out, t0, fallback = [], time.time(), []
    for i, r in df.iterrows():
        cp = os.path.join(cache_dir, r.image.rsplit(".", 1)[0] + ".png")
        if not os.path.exists(cp):
            im = cv2.imread(r.path)
            if im is None:
                out.append(None)
                continue
            try:
                proc = preprocess(im)
                if proc is None or proc.size == 0:
                    raise ValueError
            except Exception:
                h, w = im.shape[:2]
                sc = min(IN_H / h, IN_W / w)
                nh, nw = max(int(h * sc), 1), max(int(w * sc), 1)
                proc = np.full((IN_H, IN_W, 3), 255, np.uint8)
                t, l = (IN_H - nh) // 2, (IN_W - nw) // 2
                proc[t:t + nh, l:l + nw] = cv2.resize(im, (nw, nh),
                                                      interpolation=cv2.INTER_AREA)
                fallback.append(r.image)
            cv2.imwrite(cp, proc)
        out.append(cp)
        if (i + 1) % 100 == 0:
            print(f"      {i + 1}/{len(df)}  ({time.time() - t0:.0f}s)")
    df = df.copy()
    df["cached"] = out
    df = df[df.cached.notna()].reset_index(drop=True)
    print(f"[2/4] preprocessed {len(df)} images -> {IN_H}x{IN_W}, aspect preserved")
    if fallback:
        print(f"      {len(fallback)} used plain-resize fallback (kept, not dropped)")
    return df


# ------------------------------------------------------------------ phase 3
class DS(Dataset):
    def __init__(self, df, stats, train):
        self.df, self.stats, self.train = df.reset_index(drop=True), stats, train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.loc[i]
        img = cv2.cvtColor(cv2.imread(r.cached), cv2.COLOR_BGR2RGB)
        side, side_known = r.y_side, r.side_known
        if self.train:
            if np.random.rand() < 0.5:
                img = cv2.flip(img, 1)
                side = 1.0 - side          # mirroring a left foot makes a right foot
            h, w = img.shape[:2]
            M = cv2.getRotationMatrix2D((w / 2, h / 2),
                                        np.random.uniform(-8, 8),
                                        np.random.uniform(0.92, 1.08))
            M[0, 2] += np.random.uniform(-10, 10)
            M[1, 2] += np.random.uniform(-14, 14)
            img = cv2.warpAffine(img, M, (w, h), borderValue=(255, 255, 255))
            img = np.clip(img.astype(np.float32) * np.random.uniform(0.85, 1.15), 0, 255)
        x = ((img.astype(np.float32) / 255.0) - MEAN) / STD
        y = torch.tensor([(r[c] - self.stats[c][0]) / self.stats[c][1] for c in REG],
                         dtype=torch.float32)
        return (torch.from_numpy(x.transpose(2, 0, 1)), y,
                torch.tensor(r.y_gender), torch.tensor(float(side)),
                torch.tensor(float(side_known)))


class Net(nn.Module):
    def __init__(self, arch="resnet50"):
        super().__init__()
        if arch == "resnet50":
            bb = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        else:
            bb = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.stem = nn.Sequential(*list(bb.children())[:-1])
        d = bb.fc.in_features
        self.drop = nn.Dropout(0.5)
        self.reg = nn.Linear(d, 3)      # height, weight, age
        self.cls = nn.Linear(d, 1)      # sex
        self.sid = nn.Linear(d, 1)      # left vs right foot

    def forward(self, x):
        f = self.drop(self.stem(x).flatten(1))
        return self.reg(f), self.cls(f).squeeze(1), self.sid(f).squeeze(1)




def run_fold(tr, va, args):
    stats = {c: (tr[c].mean(), tr[c].std() or 1.0) for c in REG}
    dtr = DataLoader(DS(tr, stats, True), args.bs, shuffle=True, num_workers=2,
                     pin_memory=True, drop_last=len(tr) > args.bs)
    dtr_ev = DataLoader(DS(tr, stats, False), args.bs, num_workers=2)
    dva = DataLoader(DS(va, stats, False), args.bs, num_workers=2, pin_memory=True)

    net = Net(args.arch).to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=args.lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, args.lr, args.epochs * max(len(dtr), 1))
    pos = float(tr.y_gender.mean())
    bce = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor((1 - pos) / max(pos, 1e-6), device=DEV))
    bce_raw = nn.BCEWithLogitsLoss(reduction='none')   # masked: side is unknown for some
    hub = nn.HuberLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=(DEV == "cuda"))

    hist = {'epoch': [], 'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best, state, best_ep = 1e9, None, 0

    def loss_fn(pr, pg, ps, y, g, s, m):
        l = hub(pr, y) + 0.8 * bce(pg, g)
        if m.sum() > 0:                     # only score side where it is known
            l = l + 0.5 * (bce_raw(ps, s) * m).sum() / m.sum()
        return l

    def evaluate(loader):
        net.eval(); tot = n = 0.0; corr = seen = 0
        with torch.no_grad():
            for x, y, g, s, m in loader:
                x, y, g, s, m = [t.to(DEV) for t in (x, y, g, s, m)]
                with torch.amp.autocast("cuda", enabled=(DEV == "cuda")):
                    pr, pg, ps = net(x)
                    l = loss_fn(pr, pg, ps, y, g, s, m)
                tot += float(l) * len(y); n += len(y)
                corr += int(((torch.sigmoid(pg) > .5).float() == g).sum()); seen += len(g)
        return tot / max(n, 1), corr / max(seen, 1)

    for ep in range(args.epochs):
        net.train(); run = cnt = 0.0
        for x, y, g, s, m in dtr:
            x, y, g, s, m = [t.to(DEV, non_blocking=True) for t in (x, y, g, s, m)]
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(DEV == "cuda")):
                pr, pg, ps = net(x)
                l = loss_fn(pr, pg, ps, y, g, s, m)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update(); sch.step()
            run += float(l) * len(y); cnt += len(y)

        tl = run / max(cnt, 1)
        vl, va_acc = evaluate(dva)
        _, tr_acc = evaluate(dtr_ev)
        hist['epoch'].append(ep + 1); hist['train_loss'].append(tl)
        hist['val_loss'].append(vl); hist['train_acc'].append(tr_acc)
        hist['val_acc'].append(va_acc)
        if vl < best:
            best, best_ep = vl, ep + 1
            state = {k: t.cpu().clone() for k, t in net.state_dict().items()}
        if (ep + 1) % 10 == 0:
            print(f'      ep {ep+1:3d}  train {tl:.4f}  val {vl:.4f}  val_acc {va_acc:.3f}')

    print(f'      best epoch {best_ep}/{args.epochs}  (val loss {best:.4f})')
    net.load_state_dict(state); net.eval()

    def predict(loader):
        P, G, S = [], [], []
        with torch.no_grad():
            for x, y, g, s, m in loader:
                x = x.to(DEV)
                pr1, pg1, ps1 = net(x)
                pr2, pg2, ps2 = net(torch.flip(x, dims=[3]))
                P.append(((pr1 + pr2) / 2).float().cpu())
                G.append(((torch.sigmoid(pg1) + torch.sigmoid(pg2)) / 2).float().cpu())
                # a mirrored image swaps the side, so invert the flipped prediction
                S.append(((torch.sigmoid(ps1) + (1 - torch.sigmoid(ps2))) / 2).float().cpu())
        return torch.cat(P).numpy(), torch.cat(G).numpy(), torch.cat(S).numpy()

    Pv, Gv, Sv = predict(dva)
    Pt, Gt, St = predict(dtr_ev)
    unz = lambda P: {c: P[:, i] * stats[c][1] + stats[c][0] for i, c in enumerate(REG)}
    return unz(Pv), Gv, Sv, unz(Pt), Gt, St, stats, net, hist, best_ep


## 5. Preprocess

Crop to the sheet, blank the handwriting band, resize to 448×320 keeping aspect ratio. Cached, so it runs once.

In [ ]:
df = cache_all(df, args.cache)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 6, figsize=(16, 5))
for a_, (_, r) in zip(ax, df.sample(min(6, len(df)), random_state=0).iterrows()):
    a_.imshow(cv2.cvtColor(cv2.imread(r.cached), cv2.COLOR_BGR2RGB))
    a_.set_title(f"{r.gender[0]} {r.age:.0f}y {r.side}", fontsize=9); a_.axis('off')
plt.tight_layout(); plt.show()


## 6. Train — 5-fold, grouped by `cv_group`

In [ ]:
import time, json
from sklearn.model_selection import GroupKFold

groups = df.cv_group.to_numpy()
oof = {c: np.zeros(len(df)) for c in REG}
og  = np.zeros(len(df)); os_ = np.zeros(len(df))
hists, best_eps = [], []
tr_sex, tr_side, fold_stats = [], [], []      # per fold, kept separate - never summed

for k, (tr_i, va_i) in enumerate(GroupKFold(args.folds).split(df, groups=groups)):
    print(f'\nfold {k+1}/{args.folds}   train {len(tr_i)}  val {len(va_i)}')
    t0 = time.time()
    regv, gv, sv, regt, gt, st, stats, net, hist, bep = run_fold(
        df.iloc[tr_i], df.iloc[va_i], args)
    for c in REG: oof[c][va_i] = regv[c]
    og[va_i] = gv; os_[va_i] = sv
    hist['fold'] = k+1; hists.append(pd.DataFrame(hist)); best_eps.append(bep)

    # in-sample predictions, stored fold by fold so the training confusion
    # matrix can be AVERAGED over folds instead of summed
    tr_sex.append(pd.DataFrame({'fold': k+1,
                                'y': df.iloc[tr_i].y_gender.values, 'p': gt}))
    tr_side.append(pd.DataFrame({'fold': k+1,
                                 'y': df.iloc[tr_i].y_side.values,
                                 'p': st,
                                 'known': df.iloc[tr_i].side_known.values.astype(bool)}))

    # the z-score mean/std this fold used - needed to un-normalise at inference
    fold_stats.append({c: [float(stats[c][0]), float(stats[c][1])] for c in REG})
    print(f'   {time.time()-t0:.0f}s')
    torch.save(net.state_dict(), f'{args.out}/model_fold{k}.pt')

H = pd.concat(hists, ignore_index=True); H.to_csv(f'{args.out}/history.csv', index=False)
TR_SEX  = pd.concat(tr_sex,  ignore_index=True)
TR_SIDE = pd.concat(tr_side, ignore_index=True)
N_FOLDS = args.folds
print('best epochs per fold:', best_eps)


## 7. Results

In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score, r2_score,
                             confusion_matrix, precision_recall_fscore_support)
ok = df.lab_known.values.astype(bool)
res = []
print(f"{'target':12s} {'metric':6s} {'model':>9s} {'baseline':>9s}   verdict")
print('-'*58)
for c in REG:
    mae = float(np.abs(oof[c][ok]-df[c][ok]).mean()); base = float((df[c][ok]-df[c][ok].mean()).abs().mean())
    r2  = r2_score(df[c][ok], oof[c][ok])
    print(f'{c:12s} {"MAE":6s} {mae:9.2f} {base:9.2f}   {"BEATS" if mae<base*.97 else "no better"}')
    print(f'{"":12s} {"R2":6s} {r2:9.3f} {0.0:9.3f}')
    res.append(dict(target=c, mae=mae, baseline=base, r2=r2))

y = df.y_gender.values[ok]; ogk = og[ok]
acc = accuracy_score(y, ogk>.5); maj = max(y.mean(), 1-y.mean()); auc = roc_auc_score(y, ogk)
print(f'{"sex":12s} {"acc":6s} {acc:9.3f} {maj:9.3f}   {"BEATS" if acc>maj+.02 else "no better"}')
print(f'{"":12s} {"AUC":6s} {auc:9.3f} {0.5:9.3f}')
res.append(dict(target='sex', acc=acc, baseline=maj, auc=auc))

kn = df.side_known.values.astype(bool)
if kn.sum() > 10:
    ys = df.y_side.values[kn]; ps = os_[kn]
    sacc = accuracy_score(ys, ps>.5); smaj = max(ys.mean(), 1-ys.mean())
    sauc = roc_auc_score(ys, ps)
    print(f'{"left/right":12s} {"acc":6s} {sacc:9.3f} {smaj:9.3f}   {"BEATS" if sacc>smaj+.02 else "no better"}')
    print(f'{"":12s} {"AUC":6s} {sauc:9.3f} {0.5:9.3f}   (n={int(kn.sum())} with known side)')
    res.append(dict(target='side', acc=sacc, baseline=smaj, auc=sauc, n=int(kn.sum())))

pd.DataFrame(res).to_csv(f'{args.out}/metrics_table.csv', index=False)


## 8. Predictions — one row per image, all five targets

`predictions.csv` holds everything the model produced: the three regression
outputs, the sex classification, and the left/right classification, each beside
its true value and its error.

For side, `side_source` distinguishes the 336 images whose filename recorded a
side (ground truth, so `side_correct` is meaningful) from the 230 that did not
(the model's answer fills the gap).

In [ ]:
dupcol = 'is_duplicate' if 'is_duplicate' in df else 'is_duplicate_image'
if dupcol not in df:
    df[dupcol] = False
o = df[['image','cached','participant_id','cv_group','side',
        'gender','age','height_cm','weight_kg','lab_known', dupcol]].copy()
o = o.rename(columns={dupcol: 'is_duplicate'})
o = o.rename(columns={'side':'side_true'})

# --- the three regression outputs ---
for c in REG:
    o['pred_'+c]   = oof[c]
    o['err_'+c]    = oof[c] - df[c].values
    o['abserr_'+c] = np.abs(o['err_'+c])

# --- sex classification ---
o['pred_male_prob'] = og
o['pred_gender']    = np.where(og > .5, 'Male', 'Female')
o['sex_correct']    = o.pred_gender == o.gender

# --- left/right classification ---
o['pred_right_prob']  = os_
o['pred_side']        = np.where(os_ > .5, 'R', 'L')
o['side_source']      = np.where(o.side_true != 'U', 'filename', 'predicted')
o['side_confidence']  = np.abs(os_ - .5) * 2
o['side_correct']     = [None if t == 'U' else (t == p)
                         for t, p in zip(o.side_true, o.pred_side)]
o['side_final']       = np.where(o.side_true != 'U', o.side_true, o.pred_side)

# --- combined ranking, used for the best/typical/worst galleries ---
o['case_score'] = np.mean([o['abserr_'+c] / df[c].std() for c in REG], axis=0) \
                  + (~o.sex_correct).astype(float)
o = o.sort_values('case_score').reset_index(drop=True)

cols = ['image','participant_id','cv_group','lab_known','is_duplicate',
        'height_cm','pred_height_cm','err_height_cm',
        'weight_kg','pred_weight_kg','err_weight_kg',
        'age','pred_age','err_age',
        'gender','pred_gender','pred_male_prob','sex_correct',
        'side_true','pred_side','pred_right_prob','side_correct',
        'side_source','side_confidence','side_final','case_score']
o[cols].to_csv(f'{args.out}/predictions.csv', index=False)

known_side = o[o.side_source == 'filename']
print(f'predictions.csv — {len(o)} rows, {len(cols)} columns')
print(f'  sex correct       : {100*o.sex_correct.mean():.1f}%')
print(f'  side correct      : {100*known_side.side_correct.mean():.1f}%  '
      f'(on the {len(known_side)} with a recorded side)')
print(f'  sides filled in   : {int((o.side_source=="predicted").sum())}')
o[cols].head(8)


### Best, typical and worst cases

In [ ]:
def gallery(sel, title, fname):
    n = len(sel)
    fig, axes = plt.subplots(1, n, figsize=(2.7*n, 5.6)); axes = np.atleast_1d(axes)
    for a_, (_, r) in zip(axes, sel.iterrows()):
        a_.imshow(cv2.cvtColor(cv2.imread(r.cached), cv2.COLOR_BGR2RGB)); a_.axis('off')
        sm = 'ok' if r.sex_correct else 'X'
        sd = '?' if r.side_true == 'U' else ('ok' if r.side_correct else 'X')
        a_.set_title(f"{r.participant_id}\n"
                     f"ht  {r.height_cm:.0f} -> {r.pred_height_cm:.0f}\n"
                     f"wt  {r.weight_kg:.0f} -> {r.pred_weight_kg:.0f}\n"
                     f"age {r.age:.0f} -> {r.pred_age:.0f}\n"
                     f"sex {r.gender[0]} -> {r.pred_gender[0]} {sm}\n"
                     f"side {r.side_true} -> {r.pred_side} {sd}", fontsize=8)
    fig.suptitle(title, fontsize=13); plt.tight_layout()
    plt.savefig(f'{args.out}/{fname}', dpi=110); plt.show()

k = min(5, len(o)//3); mid = len(o)//2
gallery(o.head(k), 'BEST cases (lowest combined error)', 'cases_best.png')
gallery(o.iloc[mid-k//2: mid-k//2+k], 'TYPICAL cases (median error)', 'cases_typical.png')
gallery(o.tail(k).iloc[::-1], 'WORST cases (highest combined error)', 'cases_worst.png')


## 9. Figures — loss curves, confusion matrices, scatter plots

In [ ]:
fig = plt.figure(figsize=(20, 13))
R, Cc = 3, 4          # row 1 = curves, row 2 = the four matrices, row 3 = scatters

a = fig.add_subplot(R,Cc,1)
for _, g in H.groupby('fold'):
    a.plot(g.epoch, g.train_loss, 'C0', alpha=.25, lw=1); a.plot(g.epoch, g.val_loss, 'C1', alpha=.25, lw=1)
m = H.groupby('epoch')[['train_loss','val_loss']].mean()
a.plot(m.index, m.train_loss, 'C0', lw=2.5, label='train'); a.plot(m.index, m.val_loss, 'C1', lw=2.5, label='val')
a.set_xlabel('epoch'); a.set_ylabel('loss'); a.set_title('Training vs validation loss'); a.legend()

a = fig.add_subplot(R,Cc,2)
m2 = H.groupby('epoch')[['train_acc','val_acc']].mean()
a.plot(m2.index, m2.train_acc, 'C0', lw=2.5, label='train'); a.plot(m2.index, m2.val_acc, 'C1', lw=2.5, label='val')
a.axhline(maj, ls='--', c='k', lw=1, label='majority'); a.legend()
a.set_xlabel('epoch'); a.set_ylabel('sex accuracy'); a.set_title('Accuracy')

a = fig.add_subplot(R,Cc,3)
a.plot(m.index, m.val_loss - m.train_loss, 'C3', lw=2); a.axhline(0, c='k', lw=1)
a.set_xlabel('epoch'); a.set_title('Generalisation gap\n(rising = overfitting)')

a = fig.add_subplot(R,Cc,4); a.axis('off')
a.text(0, .5,
       'Confusion matrices below\n\n'
       'TRAINING  = in-sample, averaged\n'
       '            over the 5 folds\n'
       '            (not summed)\n\n'
       'HELD-OUT  = out-of-fold, each\n'
       '            image scored once by\n'
       '            a model that never\n'
       '            saw it',
       va='center', fontsize=11, family='monospace')

# ---------------------------------------------------------------- matrices
def cm_avg(parts, labels, cmap, title, ax):
    """parts: list of (y_true, p) arrays, one per fold.
    The per-fold matrices are averaged, so the counts stay on the scale of a
    single fold instead of being multiplied by the number of folds."""
    mats, accs = [], []
    for yy, pp in parts:
        if len(yy) == 0: continue
        pr = (pp > .5).astype(int)
        mats.append(confusion_matrix(yy, pr, labels=[0,1]).astype(float))
        accs.append(accuracy_score(yy, pr))
    cm = np.mean(mats, axis=0); acc = float(np.mean(accs))
    ax.imshow(cm, cmap=cmap)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i,j]:.0f}', ha='center', va='center', fontsize=14,
                    color='white' if cm[i,j] > cm.max()/2 else 'black')
    ax.set_xticks([0,1], [f'pred {labels[0]}', f'pred {labels[1]}'])
    ax.set_yticks([0,1], [f'true {labels[0]}', f'true {labels[1]}'])
    ax.set_title(f'{title}\nacc {acc:.3f}')
    return cm, acc

# --- 1. sex, in-sample (mean of the five folds)
cm_avg([(g.y.values, g.p.values) for _, g in TR_SEX.groupby('fold')],
       ('F','M'), 'Blues', 'SEX - TRAINING (in-sample)\nmean of 5 folds',
       fig.add_subplot(R,Cc,5))

# --- 2. sex, held-out
cm_avg([(y, ogk)], ('F','M'), 'Blues',
       f'SEX - HELD-OUT (out-of-fold)\nbaseline {maj:.3f}',
       fig.add_subplot(R,Cc,6))

# --- 3. left/right, in-sample (mean of the five folds)
TR_SIDE['known'] = TR_SIDE['known'].astype(bool)   # must be bool, not 1.0/0.0
side_parts = [(gg[gg.known].y.values, gg[gg.known].p.values)
              for _, gg in TR_SIDE.groupby('fold')]
side_parts = [p for p in side_parts if len(p[0]) > 0]
ax = fig.add_subplot(R,Cc,7)
if side_parts:
    cm_avg(side_parts, ('L','R'), 'Greens',
           'LEFT/RIGHT - TRAINING (in-sample)\nmean of 5 folds', ax)
else:
    ax.axis('off'); ax.set_title('LEFT/RIGHT - TRAINING\n(no known sides)')

# --- 4. left/right, held-out
ax = fig.add_subplot(R,Cc,8)
if kn.sum() > 10:
    cm_avg([(ys, ps)], ('L','R'), 'Greens',
           f'LEFT/RIGHT - HELD-OUT (out-of-fold)\nbaseline {smaj:.3f}  n={int(kn.sum())}', ax)
else:
    ax.axis('off'); ax.set_title('LEFT/RIGHT - HELD-OUT\n(too few known sides)')

# ---------------------------------------------------------------- scatters
for i, c in enumerate(REG):
    a = fig.add_subplot(R,Cc,9+i)
    tv, pv = df[c].values[ok], oof[c][ok]
    a.scatter(tv, pv, s=14, alpha=.55)
    lo, hi = tv.min(), tv.max(); a.plot([lo,hi],[lo,hi],'r--',lw=1)
    a.set_xlabel(f'true {c}'); a.set_ylabel(f'predicted {c}')
    a.set_title(f'{c}\nMAE {np.abs(pv-tv).mean():.2f}  R2 {r2_score(tv,pv):.3f}')

plt.tight_layout(); plt.savefig(f'{args.out}/figures.png', dpi=110); plt.show()
print('saved: metrics_table.csv, history.csv, figures.png, cases_*.png')


## 10. Comparing against v1

v1 dropped 204 duplicated images and scored **sex AUC 0.693, height MAE 5.71,
weight 8.55, age 5.22** on 305 images.

This run keeps all 566. If the scores come out *better*, be careful before
celebrating — the duplicates carry contradictory labels, so a large jump is
more likely to mean the grouping missed a leak than that the extra data
helped. Compare `cv_group` counts against the duplicate count if that happens.

The clean fix remains re-downloading with the authenticated script, which
removes the duplicates at source rather than working around them.

## 11. Export one inference bundle

Everything the interface needs in a single file: the five folds' weights, the
z-score mean/std each fold used (without these a prediction cannot be turned
back into cm / kg / years), and the preprocessing constants.

Download **`footprint_bundle.pt`** from the Output tab when the run finishes.


In [ ]:
bundle = {
    'arch':        args.arch,
    'folds':       args.folds,
    'in_h':        IN_H,
    'in_w':        IN_W,
    'mean':        list(map(float, np.ravel(MEAN))),
    'std':         list(map(float, np.ravel(STD))),
    'reg_targets': list(REG),
    'fold_stats':  fold_stats,      # per fold: {target: [mean, std]}
    'state_dicts': [torch.load(f'{args.out}/model_fold{k}.pt', map_location='cpu')
                    for k in range(args.folds)],
}
torch.save(bundle, f'{args.out}/footprint_bundle.pt')
import os as _os
mb = _os.path.getsize(f'{args.out}/footprint_bundle.pt') / 1e6
print(f'footprint_bundle.pt  {mb:.0f} MB  ({args.folds} folds)')
print('fold z-score stats:')
for k, st in enumerate(fold_stats):
    print(' ', k, {c: [round(v,2) for v in st[c]] for c in REG})
